# Dependencias

In [ ]:
import ctypes
ctypes.CDLL("/home/gcamilo/envs/wrf312/lib/libgomp.so.1", mode=ctypes.RTLD_GLOBAL)
import sys
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
import glob
import random
import pandas as pd
import copy
import re
import wrf
import cartopy.feature as cfeature
import cartopy.crs as crs
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from tqdm.auto import tqdm
from datetime import datetime
from IPython.display import display
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from skimage.metrics import structural_similarity as ssim
from wrf import getvar, get_cartopy, latlon_coords, to_np
from netCDF4 import Dataset as ncDataset
from scipy.ndimage import gaussian_filter1d

ruta_padre = os.path.abspath('..')
if ruta_padre not in sys.path:
    sys.path.append(ruta_padre)

CROP     = 1
# Rutas a las dos carpetas de salidas WRF (separadas por espacio en disco)
dir_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
dir_2019 = "../salidasWRF2019"
archivos = sorted(
    glob.glob(os.path.join(dir_2018, "wrfout_d01_*")) +
    glob.glob(os.path.join(dir_2019, "wrfout_d01_*"))
)
with ncDataset(archivos[0]) as ds0:
    ref = getvar(ds0, "LIGHTDENS")
    CART_PROJ = get_cartopy(ref)
    LATS, LONS = latlon_coords(ref)
LATS, LONS = to_np(LATS)[CROP:-CROP, CROP:-CROP], to_np(LONS)[CROP:-CROP, CROP:-CROP]

from modulos.tormentaDataset import TormentaDataset

# Chequeamos si hay GPU disponible para acelerar el entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
print("=== DIAGNÓSTICO DE PYTORCH Y CUDA ===")
print("Versión de PyTorch:", torch.__version__)
print("Versión de CUDA instalada en PyTorch:", torch.version.cuda)
print("¿CUDA disponible?:", torch.cuda.is_available())
print("Cantidad de GPUs detectadas:", torch.cuda.device_count())
print("=====================================\n")

if torch.cuda.is_available():
    try:
        device = torch.device("cuda")
        # Forzamos una operación matemática en la placa para validar el puente físico
        torch.zeros(1).to(device)
        print("✅ ¡Conexión exitosa a la GPU!")
        print(f"Propiedades del hardware asignado: {torch.cuda.get_device_name(0)}")
    except RuntimeError as e:
        print("❌ Error al interactuar con el hardware:", e)
else:
    print("⚠️ CUDA no está disponible en absoluto para este binario de PyTorch.")

In [ ]:
# Para REPRODUCIBILIDAD

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# U-Net Comb V2

In [ ]:
class UNet_WRF(nn.Module):
    """
    U-Net con profundidad y ancho configurables.

    Parámetros
    ----------
    in_channels : int, default 6
        Cantidad de canales de entrada (variables WRF).
    n_niveles : int, default 2
        Cantidad de niveles de encoder/decoder (pooling + upsampling).
        La arquitectura original tenía 2 niveles.
    n_filtros_base : int, default 32
        Cantidad de filtros en el primer nivel del encoder. Cada nivel
        siguiente duplica esta cantidad (32→64→128→...), igual que el
        bottleneck, que duplica una vez más el último nivel.
    kernel_sizes : list[int] o None
        Un kernel_size por nivel del encoder (y su espejo en el decoder).
        Si es None, usa kernel_size=3 en todos los niveles.
        Debe tener longitud == n_niveles.
    dropout_ps : list[float] o None
        Un dropout_p por nivel del encoder (y su espejo en el decoder),
        aplicado también en el bottleneck (usa el último valor de la lista).
        Si es None, no aplica dropout en ningún nivel (comportamiento original).
        Debe tener longitud == n_niveles.
    input_size : int, default 297
        Tamaño espacial (H=W) de la imagen de entrada, usado para calcular
        el padding necesario según la profundidad elegida.
    """
    def __init__(self, in_channels=6, n_niveles=2, n_filtros_base=32,
                 kernel_sizes=None, dropout_ps=None, input_size=297):
        super(UNet_WRF, self).__init__()

        if kernel_sizes is None:
            kernel_sizes = [3] * n_niveles
        assert len(kernel_sizes) == n_niveles, \
            f"kernel_sizes debe tener {n_niveles} elementos, tiene {len(kernel_sizes)}"

        if dropout_ps is None:
            dropout_ps = [0.0] * n_niveles
        assert len(dropout_ps) == n_niveles, \
            f"dropout_ps debe tener {n_niveles} elementos, tiene {len(dropout_ps)}"
        
        self.n_niveles = n_niveles
        self.dropout_ps = dropout_ps

        # --- Padding espacial dinámico según profundidad ---
        divisor = 2 ** n_niveles
        size_ajustado = ((input_size // divisor) + 1) * divisor
        pad_total = size_ajustado - input_size
        # Repartimos el padding: menos a la izquierda/arriba, más a la derecha/abajo
        self.pad_izq_arr = pad_total // 2
        self.pad_der_aba = pad_total - self.pad_izq_arr
        self.input_size = input_size

        # --- Canales por nivel
        canales = [n_filtros_base * (2 ** i) for i in range(n_niveles)]
        canal_bottleneck = n_filtros_base * (2 ** n_niveles)
        # Kernel del bottleneck: reusamos el del nivel más profundo
        k_bottleneck = kernel_sizes[-1]
        dropout_bottleneck = dropout_ps[-1]

        def bloque_conv(canal_in, canal_out, k, dropout_p):
            pad = k // 2
            capas = [
                nn.Conv2d(canal_in, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
                nn.Conv2d(canal_out, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
            ]
            if dropout_p > 0:
                capas.append(nn.Dropout2d(p=dropout_p))
            return nn.Sequential(*capas)
        
        # --- ENCODER (dinámico) ---
        self.enc_blocks = nn.ModuleList()
        self.pools      = nn.ModuleList()
        canal_in = in_channels
        for canal_out, k, p in zip(canales, kernel_sizes, dropout_ps):
            self.enc_blocks.append(bloque_conv(canal_in, canal_out, k, p))
            self.pools.append(nn.MaxPool2d(2, 2))
            canal_in = canal_out

        # --- BOTTLENECK ---
        self.bottleneck = bloque_conv(canal_in, canal_bottleneck, k_bottleneck, dropout_bottleneck)

        # --- DECODER (dinámico, en orden inverso al encoder) ---
        self.ups        = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        canal_in = canal_bottleneck
        for canal_skip, k, p in zip(reversed(canales), reversed(kernel_sizes), reversed(dropout_ps)):
            self.ups.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
            self.dec_blocks.append(bloque_conv(canal_in + canal_skip, canal_skip, k, p))
            canal_in = canal_skip

        # --- SALIDA ---
        self.salida  = nn.Conv2d(canal_in, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()  # SOLO para FACL

    def forward(self, x):
        x_pad = F.pad(x, (self.pad_izq_arr, self.pad_der_aba,
                           self.pad_izq_arr, self.pad_der_aba))

        # --- Encoder ---
        skips = []
        h = x_pad
        for bloque, pool in zip(self.enc_blocks, self.pools):
            h = bloque(h)
            skips.append(h)       # guardamos para el skip connection
            h = pool(h)

        # --- Bottleneck ---
        h = self.bottleneck(h)

        # --- Decoder ---
        for up, bloque, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            h = up(h)
            h = torch.cat([h, skip], dim=1)
            h = bloque(h)

        out = self.salida(h)
        out = self.sigmoid(out)  # SOLO para FACL

        # Recortar el padding agregado al principio
        out = out[:, :, self.pad_izq_arr:self.pad_izq_arr + self.input_size,
                         self.pad_izq_arr:self.pad_izq_arr + self.input_size]
        return out

In [ ]:
# HP

modelo_unet = UNet_WRF(
    in_channels=3, 
    n_niveles=3, 
    n_filtros_base=64,
    kernel_sizes=[5, 3, 3], 
    dropout_ps=[0.0, 0.0, 0.0]
).to(device)

In [ ]:
print("\n\n=== RESUMEN DE LA U-NET ===")
summary(modelo_unet, input_size=(1, 3, 297, 297), 
        col_names=["input_size", "output_size", "num_params"])

# Entrenamiento

In [ ]:
# ==========================================
# FUNCION DE PÉRDIDA
# ==========================================
class HybridFourierLossV2(nn.Module):
    """
    Variante de FACL con tres ramas:
        - WMSE  (seleccionada con probabilidad creciente P(t))
        - FAL   (seleccionada con prob (1-P(t))/2 cuando no se elige WMSE)
        - FCL   (seleccionada con prob (1-P(t))/2 cuando no se elige WMSE)

    Conforme avanza el entrenamiento, el modelo tiende a optimizar
    WMSE, mientras FAL y FCL se usan cada vez menos.
    En validación se usa una mezcla fija 1/3 de cada una.

    Parámetros
    ----------
    min_prob, max_prob, prob_slope : float
        Controlan la transición hacia WMSE (igual que RandomFourierLoss).
    peso_positivo : float
        Peso de los píxeles activos en WMSE. Default 20.
    min_y, max_y : float
        Estadísticos de normalización del target, necesarios para
        identificar píxeles activos dentro de WMSE.
    """
    def __init__(self, alpha=0.5, beta=0.5,
                 min_prob=0.0, max_prob=0.7, prob_slope=1e-5,
                 peso_positivo=20.0, min_y=0.0, max_y=1.0):
        super(HybridFourierLossV2, self).__init__()
        self.alpha         = alpha
        self.beta          = beta
        self.min_prob      = min_prob
        self.max_prob      = max_prob
        self.prob_slope    = prob_slope
        self.peso_positivo = peso_positivo
        self.step          = 0
        self.min_y         = min_y
        self.max_y         = max_y
        # Empezamos en 0 → sigma=1 → sin sesgo inicial entre términos — se aprenden junto con la red.
        self.log_var_wmse = nn.Parameter(torch.tensor(0.0))
        self.log_var_fal  = nn.Parameter(torch.tensor(0.0))
        self.log_var_fcl  = nn.Parameter(torch.tensor(0.0))

    def _ponderar(self, loss_val, log_var):
        """
        Aplica Uncertainty Weighting a un término de pérdida individual.
        
        Parámetros
        ----------
        loss_val : Tensor escalar
            El valor escalar de la función de pérdida original calculada 
            (por ejemplo, el resultado bruto de WMSE, FAL o FCL).
        
        log_var : nn.Parameter (Tensor escalar)
            El logaritmo de la varianza asociado a este término de pérdida. 
            Es un parámetro entrenable de la red que indica qué tan 
            "ruidosa" o incierta es esta tarea.

        Returns
        -------
        Tensor escalar
            El valor de la pérdida ponderada.
        """
        precision = torch.exp(-log_var)
        return precision * loss_val + log_var

    def fcl(self, fft_pred, fft_truth, eps=1e-8):
        """
        Fourier Correlation Loss (FCL).

        Mide la correlación normalizada entre las transformadas de Fourier
        de la predicción y del ground truth.

        Matemáticamente, para cada imagen i del batch:

            FCL_i = 1 - Re(<F_pred_i, conj(F_truth_i)>) /
                        (||F_pred_i|| · ||F_truth_i||)

        donde Re(·) extrae la parte real del producto interno complejo.
        El valor final es el promedio de FCL_i sobre todas las imágenes del batch.

        Esta pérdida depende de la estructura global de la imagen
        (fase + organización espacial), pero es prácticamente
        invariante ante cambios globales de intensidad.

        Parámetros
        ----------
        fft_pred : Tensor complejo
            FFT de la predicción.

        fft_truth : Tensor complejo
            FFT del ground truth.

        eps : float, default 1e-8
            Constante de estabilidad numérica para evitar división por cero.

        Returns
        -------
        Tensor escalar
            Valor promedio de la Fourier Correlation Loss sobre el batch.
        """
        dims        = tuple(range(1, fft_pred.dim()))
        numerador   = torch.sum((fft_pred * torch.conj(fft_truth)).real, dim=dims)
        norm_pred   = torch.sqrt(torch.sum(fft_pred.abs() ** 2, dim=dims))
        norm_truth  = torch.sqrt(torch.sum(fft_truth.abs() ** 2, dim=dims))
        correlacion = numerador / (norm_pred * norm_truth + eps)
        return (1 - correlacion).mean()

    def fal(self, fft_pred, fft_truth):
        """
        Fourier Amplitude Loss (FAL).

        Compara únicamente el espectro de amplitudes de ambas imágenes,
        ignorando completamente la fase.

        Matemáticamente:

            FAL = mean((|F_pred| - |F_gt|)^2)

        Esta pérdida favorece que la distribución de frecuencias sea la
        correcta, produciendo imágenes más nítidas. Sin embargo, por sí
        sola no puede reconstruir correctamente la estructura espacial,
        ya que la fase del ground truth no interviene en el gradiente.

        Parámetros
        ----------
        fft_pred : Tensor complejo
            FFT de la predicción.

        fft_truth : Tensor complejo
            FFT del ground truth.

        Returns
        -------
        Tensor escalar
            Valor de la Fourier Amplitude Loss.
        """
        return ((fft_pred.abs() - fft_truth.abs()) ** 2).mean()

    def wmse(self, pred, gt):
        """
        MSE ponderado: penaliza con mayor peso los píxeles donde hay actividad
        eléctrica real (y_real > 0), para compensar el fuerte desbalance espacial.
        Tanto pred como gt se reciben en escala normalizada [0,1].
        """
        # Desnormalizamos gt completamente para identificar píxeles activos
        gt_log  = gt * (self.max_y - self.min_y) + self.min_y  # [0,1] → log(y+0.1)
        gt_real = torch.clamp(torch.exp(gt_log) - 0.1, min=0)  # log → LIGHTDENS real

        # El error se calcula en escala [0,1] (gradiente estable)
        diferencia = (pred - gt) ** 2
        pesos = torch.where(gt_real > 0,
                            torch.tensor(self.peso_positivo, device=gt.device),
                            torch.tensor(1.0, device=gt.device))
        return torch.mean(diferencia * pesos)

    def forward(self, pred, gt, mode='train'):
        pred_sq = pred.squeeze(1)   # (N,1,H,W) → (N,H,W)
        gt_sq   = gt.squeeze(1)

        fft_pred = torch.fft.fftn(pred_sq, dim=[-1, -2], norm='ortho')
        fft_gt   = torch.fft.fftn(gt_sq,   dim=[-1, -2], norm='ortho')

        H, W   = pred_sq.shape[-2:]
        weight = np.sqrt(H * W)

        prob_t = (self.step * self.prob_slope) * (self.max_prob - self.min_prob) + self.min_prob
        prob_t = min(prob_t, self.max_prob)

        if mode == 'val':
            l_wmse = self.wmse(pred, gt)
            l_fal  = self.fal(fft_pred, fft_gt) * weight
            l_fcl  = self.fcl(fft_pred, fft_gt) * weight
            loss = (self._ponderar(l_wmse, self.log_var_wmse) +
                    self._ponderar(l_fal,  self.log_var_fal)  +
                    self._ponderar(l_fcl,  self.log_var_fcl)) / 3.0
        else:
            if np.random.rand() < prob_t:
                loss = self._ponderar(self.wmse(pred, gt), self.log_var_wmse)
            else:
                if np.random.rand() < 0.5:
                    loss = self._ponderar(self.fal(fft_pred, fft_gt) * weight, self.log_var_fal)
                else:
                    loss = self._ponderar(self.fcl(fft_pred, fft_gt) * weight, self.log_var_fcl)
            self.step += 1
                
        return loss

In [ ]:
# =====================================================================
# DATASETS
# =====================================================================

ruta_train = '../dataset/train'
ruta_val   = '../dataset/val'
ruta_stats = '../dataset/stats.npz'

print("Mapeando archivos y cargando estadísticas globales...")
train_dataset = TormentaDataset(split_dir=ruta_train, stats_path=ruta_stats)
val_dataset   = TormentaDataset(split_dir=ruta_val,   stats_path=ruta_stats)
print(f"-> Train: {len(train_dataset)} muestras | Val: {len(val_dataset)} muestras")
print(f"-> min_y: {train_dataset.min_y:.4f} | max_y: {train_dataset.max_y:.4f}")

vacias = 0
for _, y in DataLoader(train_dataset, batch_size=64, shuffle=False):
    vacias += (y.flatten(1).amax(dim=1) < 1e-6).sum().item()
print(f"-> Targets vacíos en train: {vacias} / {len(train_dataset)} ({100*vacias/len(train_dataset):.2f}%)")

In [ ]:
# =====================================================================
# HIPERPARÁMETROS (OPTIMIZADOS)
# =====================================================================

# --- Modelo ---
NOMBRE_MODELO = 'U-Net Comb V2 AvgPool'

# --- Optimización ---
LEARNING_RATE = 2.5e-4
EPOCHS        = 200
PATIENCE      = 20
BATCH_SIZE    = 16

# --- HybridFourierLossV2 ---
ALPHA         = 0.5
BETA          = 0.5
MIN_PROB      = 0.0
MAX_PROB      = 0.7
PROB_SLOPE    = 2e-5
PESO_POSITIVO = 17.53

# --- Instanciación ---
modelo = UNet_WRF(in_channels=6, n_niveles=3, n_filtros_base=64, dropout_p=0.0, kernel_sizes=[5, 3, 3]).to(device)

criterion = HybridFourierLossV2(
    alpha=ALPHA, beta=BETA,
    min_prob=MIN_PROB, max_prob=MAX_PROB, prob_slope=PROB_SLOPE,
    peso_positivo=PESO_POSITIVO,
    min_y=train_dataset.min_y, max_y=train_dataset.max_y
).to(device)

optimizer = optim.Adam(
    list(modelo.parameters()) + list(criterion.parameters()),
    lr=LEARNING_RATE
)

# --- Historiales ---
historial_train_loss   = []
historial_val_loss     = []
historial_tiempo_epoch = []

print("==================================================")
print("  CONFIGURACIÓN DE ENTRENAMIENTO   ")
print("==================================================")
print(f"-> Modelo:         {NOMBRE_MODELO}")
print(f"-> Corriendo en:   {device}")
print(f"-> Batch Size:     {BATCH_SIZE}")
print(f"-> Épocas totales: {EPOCHS}")
print(f"-> Patience (ES):  {PATIENCE} épocas")
print(f"-> Learning Rate:  {LEARNING_RATE}")
print(f"-> Función Loss:   RandomFourierLoss (min_prob={MIN_PROB}, max_prob={MAX_PROB}, prob_slope={PROB_SLOPE}, peso_positivo={PESO_POSITIVO})")
print("==================================================")

In [ ]:
# ==========================================
# DATALOADERS
# ==========================================
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"-> Batches por época (Train): {len(train_loader)}")
print(f"-> Batches por época (Val):   {len(val_loader)}")

In [ ]:
# ==========================================
# BUCLE DE ENTRENAMIENTO
# ==========================================
CARPETA_MODELOS = '../modelos'
mejor_val_loss = float('inf')
epocas_sin_mejora = 0
mejor_pesos = None
mejor_epoca = 0

os.makedirs(CARPETA_MODELOS, exist_ok=True)
fecha = datetime.now().strftime("%Y%m%d_%H%M")
ruta_guardado = os.path.join(CARPETA_MODELOS, f"{fecha}_{NOMBRE_MODELO}.pth")

print(f"--- INICIANDO ENTRENAMIENTO (máx. {EPOCHS} épocas, patience={PATIENCE}) ---")
tiempo_inicio_total = time.time()
torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # ==========================================
    # FASE DE ENTRENAMIENTO
    # ==========================================
    modelo.train() 
    train_loss_acumulada = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Train]", leave=False)
    
    for batch_X, batch_y in train_pbar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()                         
        predicciones = modelo(batch_X)         
        loss = criterion(predicciones, batch_y, mode='train')       
        loss.backward()                               
        optimizer.step()                              
        
        train_loss_acumulada += loss.item() * batch_X.size(0)
        
        # Actualizamos el texto a la derecha de la barra para ver el loss en tiempo real
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_train_loss = train_loss_acumulada / len(train_loader.dataset)
    historial_train_loss.append(epoch_train_loss)
    
    # ==========================================
    # FASE DE VALIDACIÓN
    # ==========================================
    modelo.eval()
    val_loss_acumulada = 0.0
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Val]  ", leave=False)
    
    with torch.no_grad(): 
        for batch_X, batch_y in val_pbar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            predicciones = modelo(batch_X)
            loss = criterion(predicciones, batch_y, mode='val')
            
            val_loss_acumulada += loss.item() * batch_X.size(0)
            val_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    epoch_val_loss = val_loss_acumulada / len(val_loader.dataset)
    historial_val_loss.append(epoch_val_loss)
    tiempo_epoch = time.time() - start_time
    historial_tiempo_epoch.append(tiempo_epoch)

    # ==========================================
    # EARLY STOPPING
    # ==========================================
    if epoch_val_loss < mejor_val_loss:
        mejor_val_loss = epoch_val_loss
        epocas_sin_mejora = 0
        mejor_epoca = epoch + 1
        # Guardamos una copia en memoria de los mejores pesos
        mejor_pesos = copy.deepcopy(modelo.state_dict())
    else:
        epocas_sin_mejora += 1
    
    # ==========================================
    # GUARDADO
    # ==========================================
    torch.save({
        'model_state_dict': modelo.state_dict(),
        'best_model_state_dict': mejor_pesos,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict':  scheduler.state_dict(),
        'criterion_state_dict': criterion.state_dict(),
        'historial_train_loss': historial_train_loss,
        'historial_val_loss': historial_val_loss,
        'historial_tiempo_epoch': historial_tiempo_epoch,
        'mejor_epoca': mejor_epoca,
        'mejor_val_loss': mejor_val_loss,
        'epochs_entrenadas': epoch + 1,
        'lr': LEARNING_RATE,
        'patience': PATIENCE,
    }, ruta_guardado)

    # ==========================================
    # SCHEDULER STEP
    # ==========================================
    scheduler.step()
    lr_actual = optimizer.param_groups[0]['lr']
    
    # ==========================================
    # REPORTE DE LA ÉPOCA
    # ==========================================
    tiempo_epoch = time.time() - start_time
    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Tiempo: {tiempo_epoch:.2f}s | "
          f"Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f} | "
          f"Sin mejora: {epocas_sin_mejora}/{PATIENCE}")
    
    # ==========================================
    # CORTE POR EARLY STOPPING
    # ==========================================
    if epocas_sin_mejora >= PATIENCE:
        print(f"\nEarly stopping activado en la época {epoch+1}.")
        print(f"Mejor val_loss: {mejor_val_loss:.6f} (época {mejor_epoca})")
        break

# ==========================================
# RESTAURAR MEJORES PESOS
# ==========================================
modelo.load_state_dict(mejor_pesos)
print(f"\nPesos restaurados a la mejor época ({mejor_epoca}).")

# ==========================================
# REPORTE FINAL
# ==========================================
tiempo_fin_total = time.time()
segundos_totales = tiempo_fin_total - tiempo_inicio_total
horas_totales = segundos_totales / 3600
horas_enteras = int(horas_totales)
minutos_enteros = int((segundos_totales % 3600) / 60)

print("\n¡Entrenamiento finalizado exitosamente!")
print(f"Modelo final disponible en: {ruta_guardado}")
print(f"Tiempo total: {horas_totales:.2f} horas ({horas_enteras}h {minutos_enteros}m)")
print(f"Tiempo promedio por época: {np.mean(historial_tiempo_epoch):.1f}s")
print(f"Mejor época: {mejor_epoca} | Mejor val_loss: {mejor_val_loss:.6f}")

# Optimización

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna'], capture_output=True, text=True)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

## Etapa A — Búsqueda de Arquitectura 

In [ ]:
import optuna

def objective_arquitectura(trial):
    tiempo_inicio_trial = time.time()

    # ==========================================
    # HIPERPARÁMETROS DE ARQUITECTURA
    # ==========================================
    n_niveles      = trial.suggest_int('n_niveles', 1, 4)
    n_filtros_base = trial.suggest_categorical('n_filtros_base', [16, 32, 64, 128])
    dropout_p      = trial.suggest_float('dropout_p', 0.0, 0.5)

    kernel_sizes = [
        trial.suggest_categorical(f'kernel_nivel_{i}', [3, 5, 7])
        for i in range(n_niveles)
    ]

    # ==========================================
    # HIPERPARÁMETROS DE ENTRENAMIENTO (FIJOS)
    # ==========================================
    LR             = 1e-4
    BATCH_SIZE     = 16
    MAX_PROB       = 0.75
    PROB_SLOPE     = 1e-5
    PESO_POSITIVO  = 20.0
    MIN_PROB       = 0.0
    EPOCAS_TRIAL   = 30
    PATIENCE_TRIAL = 6

    # ==========================================
    # DATALOADERS
    # ==========================================
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader_trial = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                     shuffle=True, generator=generator,
                                     num_workers=0, pin_memory=True)
    val_loader_trial   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                                     shuffle=False, num_workers=0, pin_memory=True)

    # ==========================================
    # MODELO, OPTIMIZER, LOSS Y SCHEDULER
    # ==========================================
    modelo_trial = UNet_WRF(
        n_niveles=n_niveles, n_filtros_base=n_filtros_base,
        kernel_sizes=kernel_sizes, dropout_p=dropout_p
    ).to(device)

    optimizer_trial = optim.Adam(modelo_trial.parameters(), lr=LR)
    criterion_trial = HybridFourierLossV2(
        min_prob=MIN_PROB, max_prob=MAX_PROB, prob_slope=PROB_SLOPE,
        peso_positivo=PESO_POSITIVO,
        min_y=train_dataset.min_y, max_y=train_dataset.max_y
    ).to(device)
    scheduler_trial = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_trial, T_max=EPOCAS_TRIAL, eta_min=1e-7
    )

    mejor_val_loss    = float('inf')
    epocas_sin_mejora = 0

    # ==========================================
    # BUCLE DE ENTRENAMIENTO DEL TRIAL
    # ==========================================
    try:
        for epoch in range(EPOCAS_TRIAL):
            modelo_trial.train()
            for batch_X, batch_y in train_loader_trial:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_trial.zero_grad()
                pred = modelo_trial(batch_X)
                loss = criterion_trial(pred, batch_y, mode='train')
                loss.backward()
                optimizer_trial.step()

            modelo_trial.eval()
            val_loss_acumulada = 0.0
            with torch.no_grad():
                for batch_X, batch_y in val_loader_trial:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    pred = modelo_trial(batch_X)
                    loss = criterion_trial(pred, batch_y, mode='val')
                    val_loss_acumulada += loss.item() * batch_X.size(0)

            epoch_val_loss = val_loss_acumulada / len(val_loader_trial.dataset)
            # Detectar NaN/Inf y podar el trial en lugar de dejarlo completarse con basura
            if not np.isfinite(epoch_val_loss):
                print(f"[Trial {trial.number}] val_loss no finito ({epoch_val_loss}) — podando trial.")
                raise optuna.TrialPruned()

            scheduler_trial.step()

            if epoch_val_loss < mejor_val_loss:
                mejor_val_loss = epoch_val_loss
                epocas_sin_mejora = 0
            else:
                epocas_sin_mejora += 1
                if epocas_sin_mejora >= PATIENCE_TRIAL:
                    break

            trial.report(epoch_val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Epoch {epoch+1}/{EPOCAS_TRIAL} | val_loss={epoch_val_loss:.4f}")

        trial.set_user_attr('tiempo_trial_seg', time.time() - tiempo_inicio_trial)
        trial.set_user_attr('n_parametros', sum(p.numel() for p in modelo_trial.parameters()))
        return mejor_val_loss

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] OOM — se descarta este trial.")
        raise optuna.TrialPruned()

In [ ]:
study_arq = optuna.create_study(
    study_name='busqueda_arquitectura_unet',
    storage='sqlite:///../estudios/optuna_arquitectura.db',
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=7),
    load_if_exists=True
)

study_arq.optimize(objective_arquitectura, timeout=18*3600)

In [ ]:
study_arq = optuna.load_study(
    study_name='busqueda_arquitectura_unet',
    storage='sqlite:///../estudios/optuna_arquitectura.db'
)

print(f"Estudio cargado: {len(study_arq.trials)} trials en total")

In [ ]:
# =====================================================================
# RESUMEN DE LA ETAPA A — BÚSQUEDA DE ARQUITECTURA
# =====================================================================

trials_completados = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.COMPLETE]
trials_podados      = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.PRUNED]
trials_fallidos     = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.FAIL]

print("="*60)
print("RESUMEN ETAPA A — BÚSQUEDA DE ARQUITECTURA")
print("="*60)
print(f"Total de trials registrados:   {len(study_arq.trials)}")
print(f"  -> Completados exitosamente: {len(trials_completados)}")
print(f"  -> Podados (pruned):         {len(trials_podados)}")
print(f"  -> Fallidos:                 {len(trials_fallidos)}")

# --- Detectar cuántos completados terminaron en NaN ---
trials_nan = [t for t in trials_completados if t.value is not None and np.isnan(t.value)]
print(f"  -> Completados con valor NaN: {len(trials_nan)}")

print("\n" + "-"*60)
print("MEJOR TRIAL ENCONTRADO")
print("-"*60)
if len(trials_completados) > 0:
    print(f"Trial número: {study_arq.best_trial.number}")
    print(f"Val Loss:     {study_arq.best_trial.value:.6f}")
    print(f"\nHiperparámetros de arquitectura:")
    for k, v in study_arq.best_trial.params.items():
        print(f"  {k}: {v}")
    print(f"\nCantidad de parámetros del modelo: {study_arq.best_trial.user_attrs.get('n_parametros', 'N/D'):,}")
    print(f"Tiempo del trial: {study_arq.best_trial.user_attrs.get('tiempo_trial_seg', 'N/D')}")
else:
    print("⚠️ Ningún trial se completó con éxito.")

print("\n" + "-"*60)
print("TOP 5 MEJORES TRIALS (por Val Loss)")
print("-"*60)
trials_df = study_arq.trials_dataframe()
trials_df_validos = trials_df[trials_df['state'] == 'COMPLETE'].sort_values('value')
cols_mostrar = ['number', 'value'] + [c for c in trials_df_validos.columns if c.startswith('params_')]
display(trials_df_validos[cols_mostrar].head(5))

from optuna.importance import FanovaImportanceEvaluator
evaluador_fijo = FanovaImportanceEvaluator(seed=SEED)

print("\n" + "-"*60)
print("IMPORTANCIA DE HIPERPARÁMETROS")
print("-"*60)
try:
    importancias = optuna.importance.get_param_importances(study_arq, evaluator=evaluador_fijo)
    for k, v in importancias.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"No se pudo calcular importancia: {e}")

print("\n" + "-"*60)
print("TIEMPO TOTAL DE LA ETAPA")
print("-"*60)
tiempos = [t.user_attrs.get('tiempo_trial_seg', 0) for t in trials_completados]
if tiempos:
    print(f"Tiempo total acumulado en trials completados: {sum(tiempos)/3600:.2f} horas")
    print(f"Tiempo promedio por trial completado:         {np.mean(tiempos)/60:.1f} minutos")

# --- Gráfico de importancia ---
import optuna.visualization.matplotlib as vis_mpl

vis_mpl.plot_param_importances(study_arq, evaluator=evaluador_fijo)
plt.tight_layout()
plt.show()

In [ ]:
trials_df = study_arq.trials_dataframe()
plt.scatter(trials_df['user_attrs_n_parametros'], trials_df['value'], alpha=0.6)
plt.xlabel('Cantidad de parámetros')
plt.ylabel('Val Loss (mejor época del trial)')
plt.xscale('log')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
trials_df = study_arq.trials_dataframe()

# Separar completados y podados
completos = trials_df[trials_df['state'] == 'COMPLETE']
podados   = trials_df[trials_df['state'] == 'PRUNED']

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(completos['params_dropout_p'], completos['value'],
           alpha=0.6, s=40, color='#722F37', label='Completados')

# Para los podados no hay 'value' final útil, pero podemos marcar dónde cayeron en X
if len(podados) > 0:
    ax.scatter(podados['params_dropout_p'], [completos['value'].max()] * len(podados),
               alpha=0.4, s=40, color='gray', marker='x', label='Podados (sin valor final)')

# Marcar el mejor trial
mejor = trials_df.loc[trials_df['value'].idxmin()]
ax.scatter(mejor['params_dropout_p'], mejor['value'],
           s=250, marker='*', color='gold', edgecolor='black', zorder=5,
           label=f"Mejor trial ({int(mejor['number'])})")

ax.set_xlabel('dropout_p')
ax.set_ylabel('Val Loss')
ax.set_title('Val Loss vs Dropout — Todos los trials completados')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Complementario: promedio de val_loss agrupado por rangos de dropout
completos_copia = completos.copy()
completos_copia['dropout_bin'] = pd.cut(completos_copia['params_dropout_p'],
                                          bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5])
print(completos_copia.groupby('dropout_bin', observed=True)['value'].agg(['mean', 'std', 'count']))

## Etapa B — Optimización de Hiperparámetros de Entrenamiento

In [ ]:
ARQ_FINAL = dict(
    n_niveles=3,
    n_filtros_base=64,
    kernel_sizes=[5, 3, 3],
    dropout_p=0.0,
)

In [ ]:
def objective_entrenamiento(trial):
    torch.cuda.empty_cache()
    tiempo_inicio_trial = time.time()

    # ==========================================
    # HIPERPARÁMETROS DE ENTRENAMIENTO A OPTIMIZAR
    # ==========================================
    lr             = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    batch_size     = trial.suggest_categorical('batch_size', [4, 8, 16, 32])
    max_prob       = trial.suggest_float('max_prob', 0.5, 0.95)
    prob_slope     = trial.suggest_float('prob_slope', 1e-6, 1e-4, log=True)
    peso_positivo  = trial.suggest_float('peso_positivo', 5.0, 50.0)

    # ==========================================
    # CONFIGURACIÓN FIJA DE LA ETAPA B
    # ==========================================
    MIN_PROB       = 0.0
    EPOCAS_TRIAL   = 50
    PATIENCE_TRIAL = 10

    # ==========================================
    # DATALOADERS (batch_size varía por trial)
    # ==========================================
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader_trial = DataLoader(train_dataset, batch_size=batch_size,
                                     shuffle=True, generator=generator,
                                     num_workers=0, pin_memory=True)
    val_loader_trial   = DataLoader(val_dataset, batch_size=batch_size,
                                     shuffle=False, num_workers=0, pin_memory=True)

    # ==========================================
    # MODELO — arquitectura FIJA (ganadora de la Etapa A)
    # ==========================================
    modelo_trial = UNet_WRF(
        n_niveles=ARQ_FINAL['n_niveles'],
        n_filtros_base=ARQ_FINAL['n_filtros_base'],
        kernel_sizes=ARQ_FINAL['kernel_sizes'],
        dropout_p=ARQ_FINAL['dropout_p'],
    ).to(device)

    optimizer_trial = optim.Adam(modelo_trial.parameters(), lr=lr)
    criterion_trial = HybridFourierLossV2(
        min_prob=MIN_PROB, max_prob=max_prob, prob_slope=prob_slope,
        peso_positivo=peso_positivo,
        min_y=train_dataset.min_y, max_y=train_dataset.max_y
    ).to(device)
    scheduler_trial = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_trial, T_max=EPOCAS_TRIAL, eta_min=1e-7
    )

    mejor_val_loss    = float('inf')
    epocas_sin_mejora = 0

    # ==========================================
    # BUCLE DE ENTRENAMIENTO DEL TRIAL
    # ==========================================
    try:
        for epoch in range(EPOCAS_TRIAL):
            modelo_trial.train()
            for batch_X, batch_y in train_loader_trial:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_trial.zero_grad()
                pred = modelo_trial(batch_X)
                loss = criterion_trial(pred, batch_y, mode='train')
                loss.backward()
                optimizer_trial.step()

            modelo_trial.eval()
            val_loss_acumulada = 0.0
            with torch.no_grad():
                for batch_X, batch_y in val_loader_trial:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    pred = modelo_trial(batch_X)
                    loss = criterion_trial(pred, batch_y, mode='val')
                    val_loss_acumulada += loss.item() * batch_X.size(0)

            epoch_val_loss = val_loss_acumulada / len(val_loader_trial.dataset)

            # --- Chequeo de NaN/Inf ---
            if not np.isfinite(epoch_val_loss):
                print(f"[Trial {trial.number}] val_loss no finito ({epoch_val_loss}) — podando trial.")
                raise optuna.TrialPruned()

            scheduler_trial.step()

            if epoch_val_loss < mejor_val_loss:
                mejor_val_loss = epoch_val_loss
                epocas_sin_mejora = 0
            else:
                epocas_sin_mejora += 1
                if epocas_sin_mejora >= PATIENCE_TRIAL:
                    break

            trial.report(epoch_val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Epoch {epoch+1}/{EPOCAS_TRIAL} | val_loss={epoch_val_loss:.4f}")

        trial.set_user_attr('tiempo_trial_seg', time.time() - tiempo_inicio_trial)
        return mejor_val_loss

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] OOM — se descarta este trial.")
        raise optuna.TrialPruned()

In [ ]:
study_entrenamiento = optuna.create_study(
    study_name='optimizacion_entrenamiento_unet',
    storage='sqlite:///../estudios/optuna_entrenamiento.db',
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=7),
    load_if_exists=True
)

study_entrenamiento.optimize(objective_entrenamiento, timeout=18*3600)

In [ ]:
study_entrenamiento = optuna.load_study(
    study_name='optimizacion_entrenamiento_unet',
    storage='sqlite:///../estudios/optuna_entrenamiento.db'
)

print(f"Estudio cargado: {len(study_entrenamiento.trials)} trials en total")

In [ ]:
# =====================================================================
# RESUMEN DE LA ETAPA B — OPTIMIZACIÓN DE HIPERPARÁMETROS DE ENTRENAMIENTO
# =====================================================================

trials_completados = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.COMPLETE]
trials_podados      = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.PRUNED]
trials_fallidos     = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.FAIL]

print("="*60)
print("RESUMEN ETAPA B — OPTIMIZACIÓN DE ENTRENAMIENTO")
print("="*60)
print(f"Total de trials registrados:   {len(study_entrenamiento.trials)}")
print(f"  -> Completados exitosamente: {len(trials_completados)}")
print(f"  -> Podados (pruned):         {len(trials_podados)}")
print(f"  -> Fallidos:                 {len(trials_fallidos)}")

trials_nan = [t for t in trials_completados if t.value is not None and np.isnan(t.value)]
print(f"  -> Completados con valor NaN: {len(trials_nan)}")

print("\n" + "-"*60)
print("MEJOR TRIAL ENCONTRADO")
print("-"*60)
if len(trials_completados) > 0:
    print(f"Trial número: {study_entrenamiento.best_trial.number}")
    print(f"Val Loss:     {study_entrenamiento.best_trial.value:.6f}")
    print(f"\nHiperparámetros de entrenamiento:")
    for k, v in study_entrenamiento.best_trial.params.items():
        print(f"  {k}: {v}")
    print(f"\nTiempo del trial: {study_entrenamiento.best_trial.user_attrs.get('tiempo_trial_seg', 'N/D')}")
else:
    print("⚠️ Ningún trial se completó con éxito.")

print("\n" + "-"*60)
print("TOP 5 MEJORES TRIALS (por Val Loss)")
print("-"*60)
trials_df = study_entrenamiento.trials_dataframe()
trials_df_validos = trials_df[trials_df['state'] == 'COMPLETE'].sort_values('value')
cols_mostrar = ['number', 'value'] + [c for c in trials_df_validos.columns if c.startswith('params_')]
display(trials_df_validos[cols_mostrar].head(5))

from optuna.importance import FanovaImportanceEvaluator
evaluador_fijo = FanovaImportanceEvaluator(seed=SEED)

print("\n" + "-"*60)
print("IMPORTANCIA DE HIPERPARÁMETROS")
print("-"*60)
try:
    importancias_reproducibles = optuna.importance.get_param_importances(
        study_entrenamiento, evaluator=evaluador_fijo
    )
    for k, v in importancias_reproducibles.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"No se pudo calcular importancia (default): {e}")

print("\n" + "-"*60)
print("TIEMPO TOTAL DE LA ETAPA")
print("-"*60)
tiempos = [t.user_attrs.get('tiempo_trial_seg', 0) for t in trials_completados]
if tiempos:
    print(f"Tiempo total acumulado en trials completados: {sum(tiempos)/3600:.2f} horas")
    print(f"Tiempo promedio por trial completado:         {np.mean(tiempos)/60:.1f} minutos")

# --- Gráfico de importancia ---
import optuna.visualization.matplotlib as vis_mpl

vis_mpl.plot_param_importances(study_entrenamiento, evaluator=evaluador_fijo)
plt.tight_layout()
plt.show()

# Resultados

In [ ]:
def resumen_entrenamiento(checkpoint, nombre_modelo, train_loader, val_loader, batch_size):
    """
    Imprime un resumen del entrenamiento combinando datos del checkpoint
    (guardados durante el entrenamiento) con datos del DataLoader actual
    (que no se guardan en el .pth).
    """
    tiempo_total_seg = sum(checkpoint['historial_tiempo_epoch'])
    horas = int(tiempo_total_seg // 3600)
    minutos = int((tiempo_total_seg % 3600) // 60)
    tiempo_str = f"{horas}h {minutos}m" if horas > 0 else f"{minutos}m"

    print(f"-> Modelo Activo:  {nombre_modelo}")
    print(f"-> Épocas totales: {checkpoint['epochs_entrenadas']}")
    print(f"-> Learning Rate:  {checkpoint['lr']}")
    print(f"-> Batch Size:     {batch_size}")
    print(f"-> Archivos para entrenar (Train): {len(train_loader.dataset)}")
    print(f"-> Archivos para validar (Val): {len(val_loader.dataset)}")
    print(f"-> Batches por época (Train): {len(train_loader)}")
    print(f"-> Tiempo: {tiempo_str}")
    print(f"-> Mejor val_loss: {checkpoint['mejor_val_loss']:.6f} (época {checkpoint['mejor_epoca']})")
    print(f"-> Patience: {checkpoint['patience']}")

def generar_ruta_imagen(carpeta, prefijo, idx=None):
    """
    Genera la ruta de guardado. Si se provee un 'idx', lo incluye en el nombre.
    Si el archivo ya existe, le anexa un contador (_01, _02, etc.) para no pisarlo.
    """
    os.makedirs(carpeta, exist_ok=True)
    
    # Armamos el nombre base. Ej: "reconstruccion_unet_idx_42" o "curva_error_unet"
    if idx is not None:
        nombre_base = f"{prefijo}_idx_{idx}"
    else:
        nombre_base = prefijo
        
    ruta = os.path.join(carpeta, f"{nombre_base}.png")
    
    # Si no existe, usamos ese nombre directamente
    if not os.path.exists(ruta):
        return ruta
        
    # Si ya existe, empezamos a probar con un contador hasta encontrar uno libre
    contador = 1
    while True:
        ruta_con_contador = os.path.join(carpeta, f"{nombre_base}_{contador:02d}.png")
        if not os.path.exists(ruta_con_contador):
            return ruta_con_contador
        contador += 1

def graficar_curva_error(checkpoint, nombre_modelo='Modelo'):
    """
    Grafica la curva de MSE para train y validación a partir de un checkpoint cargado.
    """
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    mejor_epoca    = checkpoint['mejor_epoca']
    mejor_val_loss = checkpoint['mejor_val_loss']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epocas, historial_train, 'o-', label='Train Loss (HFACLV2)', color='steelblue')
    ax.plot(epocas, historial_val,   'o-', label='Val Loss (HFACLV2)',   color='#722F37')
    
    ax.scatter([mejor_epoca], [mejor_val_loss], s=180, marker='*',
               color='gold', edgecolor='black', zorder=5,
               label=f'Mejor época ({mejor_epoca})')
    ax.axvline(mejor_epoca, color='gray', linestyle='--', alpha=0.5)
    ax.annotate(f'{mejor_val_loss:.4f}',
                xy=(mejor_epoca, mejor_val_loss),
                xytext=(8, 10), textcoords='offset points',
                fontsize=9, fontweight='bold')
    
    ax.set_xlabel('Época')
    ax.set_ylabel('HFACLV2 Loss')
    ax.set_title(f'Curva de aprendizaje — {nombre_modelo}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"curva_error_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de error guardada en: {ruta_guardado}")
    
    plt.show()
    
    mejor_epoca = historial_val.index(min(historial_val)) + 1
    print(f"Mejor val_loss: {min(historial_val):.6f} (época {mejor_epoca})")

def graficar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', canal_entrada=0, idx=None, vmax_percentil=99):
    """
    Muestra la variable de entrada, el target real y la predicción del modelo
    para un timestep aleatorio (o el índice especificado) del dataset.

    Parámetros
    ----------
    modelo        : modelo PyTorch ya cargado y en eval mode
    dataset       : TormentaDataset (train o val)
    nombre_modelo : str, para el título
    canal_entrada : int, índice del canal a mostrar como entrada
    idx           : int o None. Si None, elige uno al azar.
    """
    VAR_NOMBRES = ['MDBZ', 'HIELO', 'AGUA', 'CTT', 'LPI', 'WMAX']

    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    modelo.eval()
    X, y = dataset[idx]
    with torch.no_grad():
        pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

    # Desnormalizar target y predicción
    y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
    y_pred = dataset.desnormalizar_y(pred_log).numpy()
    entrada = X[canal_entrada].numpy()

    # Escala basada en percentil (ignora picos extremos)
    vmax_real = np.percentile(y_real[y_real > 0], vmax_percentil) if (y_real > 0).any() else 1

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{nombre_modelo} — Timestep {idx}', fontsize=13)

    ax0 = fig.add_subplot(1, 3, 1, projection=CART_PROJ)
    ax0.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax0.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax0.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
    cf0 = ax0.contourf(LONS, LATS, entrada, transform=crs.PlateCarree(), cmap='Blues')
    ax0.set_title(f'Entrada: {VAR_NOMBRES[canal_entrada]}')
    fig.colorbar(cf0, ax=ax0, fraction=0.046, pad=0.04)

    ax1 = fig.add_subplot(1, 3, 2, projection=CART_PROJ)
    ax1.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax1.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
    cf1 = ax1.contourf(LONS, LATS, y_real, transform=crs.PlateCarree(),
                        cmap='hot_r', levels=np.linspace(0, vmax_real, 21), vmin=0, vmax=vmax_real, extend='max')
    ax1.set_title('Real: LIGHTDENS')
    fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)

    ax2 = fig.add_subplot(1, 3, 3, projection=CART_PROJ)
    ax2.add_feature(cfeature.BORDERS, linewidth=0.8)
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax2.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
    cf2 = ax2.contourf(LONS, LATS, y_pred, transform=crs.PlateCarree(),
                        cmap='hot_r', levels=np.linspace(0, vmax_real, 21), vmin=0, vmax=vmax_real, extend='max')
    ax2.set_title(f'Predicción: {nombre_modelo}')
    fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04)

    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"reconstruccion_{nombre_limpio}", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Reconstrucción guardada en: {ruta_guardado}")
    
    plt.show()

    print(f"Real  — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    print(f"Pred  — min: {y_pred.min():.2f}, max: {y_pred.max():.2f}, mean: {y_pred.mean():.4f}")


def animar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', fps=5, vmax_percentil=99):
    """
    Genera una animación comparando la predicción del modelo contra el valor real
    de LIGHTDENS para todos los timesteps del dataset (pensado para el conjunto
    de validación, que contiene un día completo).

    Parámetros
    ----------
    modelo        : modelo PyTorch ya cargado y en eval mode
    dataset       : TormentaDataset (normalmente val_dataset)
    nombre_modelo : str, para el título y el nombre de archivo
    fps           : cuadros por segundo de la animación
    """
    modelo.eval()
    n_frames = len(dataset)

    # --- 1. Calcular vmax global (vmin siempre es 0 porque LIGHTDENS no es negativo) ---
    vmax = 0.0
    predicciones_cache = []  # Cacheamos para no correr el modelo dos veces
    reales_cache = []

    for idx in tqdm(range(n_frames), desc=f"Calculando rango — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()

        reales_cache.append(y_real)
        predicciones_cache.append(y_pred)

    valores_activos = np.concatenate([r[r > 0] for r in reales_cache if (r > 0).any()])
    vmax = np.percentile(valores_activos, vmax_percentil) if len(valores_activos) > 0 else 1

    print(f"Rango LIGHTDENS: 0.00 - {vmax:.2f}")

    # --- 2. Extraer fecha del primer archivo del dataset ---
    nombre_archivo = dataset.files[0].name
    match = re.search(r'(\d{4}-\d{2}-\d{2})', nombre_archivo)
    fecha_val = match.group(1) if match else "fecha_desconocida"

    # --- 3. Preparar figura ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': CART_PROJ})
    fig.suptitle(f'{nombre_modelo} — Validación {fecha_val}', fontsize=14)

    pbar = tqdm(total=n_frames, desc="Generando frames")

    def update(frame_idx):
        for ax in axes:
            ax.clear()
            ax.add_feature(cfeature.BORDERS, linewidth=0.8)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
            ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')

        cf0 = axes[0].contourf(LONS, LATS, reales_cache[frame_idx], transform=crs.PlateCarree(),
                               cmap='hot_r', levels=np.linspace(0, vmax, 21), vmin=0, vmax=vmax, extend='max')
        axes[0].set_title(f'Real: LIGHTDENS — frame {frame_idx}')

        cf1 = axes[1].contourf(LONS, LATS, predicciones_cache[frame_idx], transform=crs.PlateCarree(),
                               cmap='hot_r', levels=np.linspace(0, vmax, 21), vmin=0, vmax=vmax, extend='max')
        axes[1].set_title(f'Predicción: {nombre_modelo}')

        pbar.update(1)
        return cf0, cf1

    # --- 4. Colorbar único y animación ---
    cf_inicial_real, cf_inicial_pred = update(0)
    fig.colorbar(cf_inicial_real, ax=axes[0], label='LIGHTDENS (flash/columna)', fraction=0.046, pad=0.04)
    fig.colorbar(cf_inicial_pred, ax=axes[1], label='LIGHTDENS (flash/columna)', fraction=0.046, pad=0.04)

    ani = animation.FuncAnimation(fig, update,
                                   frames=range(1, n_frames),
                                   interval=200)

    # --- 5. Guardado ---
    carpeta_destino = "../videos&gifs"
    os.makedirs(carpeta_destino, exist_ok=True)
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    nombre_archivo_salida = f"reconstruccion_{nombre_limpio}_{fecha_val}.gif"
    ruta_final = os.path.join(carpeta_destino, nombre_archivo_salida)

    ani.save(ruta_final, writer="pillow", fps=fps)
    pbar.close()
    print(f"Animación guardada en {ruta_final}")
    plt.show()

def mostrar_historial_tabular(checkpoint):
    """
    Toma el checkpoint de un modelo y muestra una tabla interactiva con 
    el historial de pérdidas (Train y Val) época por época.
    Resalta en verde la época con el menor error de validación.
    """
    # 1. Extraemos las listas del checkpoint
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    
    # 2. Armamos un DataFrame de Pandas
    df = pd.DataFrame({
        'Época': epocas,
        'Train Loss (HFACLV2)': historial_train,
        'Val Loss (HFACLV2)': historial_val
    })
    
    # 3. Calculamos la diferencia con la época anterior para ver si mejoró
    mejor_hasta_ayer = df['Val Loss (HFACLV2)'].shift(1).cummin().fillna(float('inf'))
    df['¿Mejoró?'] = (df['Val Loss (HFACLV2)'] < mejor_hasta_ayer).apply(lambda x: '🟢 Sí' if x else '🔴 No')
    df.loc[0, '¿Mejoró?'] = '🟢 Sí'
    
    # 4. Buscamos cuál fue la mejor época absoluta
    mejor_val_loss = min(historial_val)
    
    # 5. Función para pintar la fila ganadora de verde
    def resaltar_mejor_epoca(fila):
        if fila['Val Loss (HFACLV2)'] == mejor_val_loss:
            return ['background-color: #d4edda; color: #155724; font-weight: bold'] * len(fila)
        else:
            return [''] * len(fila)
            
    # 6. Aplicamos el formato (dejamos 6 decimales) y mostramos la tabla
    tabla_estilizada = df.style.apply(resaltar_mejor_epoca, axis=1) \
                               .format({'Train Loss (HFACLV2)': "{:.6f}", 
                                        'Val Loss (HFACLV2)': "{:.6f}"}) \
                               .hide(axis="index")
    
    print("HISTORIAL DE ENTRENAMIENTO TABULAR")
    print("-" * 40)
    display(tabla_estilizada)

    return df

def calcular_data_range_global(dataset):
    """
    Calcula el rango (max - min) de LIGHTDENS real sobre TODO el dataset.
    """
    max_global = float("-inf")
    min_global = float("inf")
    for idx in tqdm(range(len(dataset)), desc="Calculando data_range global"):
        _, y = dataset[idx]
        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        max_global = max(max_global, y_real.max())
        min_global = min(min_global, y_real.min())
    return max_global - min_global

def calcular_metricas_timestep(y_real, y_pred, data_range=None):
    """
    Calcula un set de métricas de verificación comparando la predicción
    contra LIGHTDENS real, para un único timestep.

    Parámetros
    ----------
    y_real, y_pred : np.ndarray 2D

    Returns
    -------
    dict con: mse, rmse, ssim
    """
    error = y_real - y_pred
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))

    if data_range is None:
        data_range = max(y_real.max(), y_pred.max()) - min(y_real.min(), y_pred.min())
    valor_ssim = ssim(y_real, y_pred, data_range=data_range) if data_range > 0 else 1.0
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim}

def evaluar_dataset(modelo, dataset, data_range=None):
    """
    Recorre todo el dataset, calcula las métricas de verificación para cada
    timestep y las devuelve en un DataFrame (una fila por timestep).
    """
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    for idx in tqdm(range(len(dataset)), desc="Calculando métricas"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()

        metricas = calcular_metricas_timestep(y_real, y_pred, data_range=data_range)
        metricas['idx'] = idx
        registros.append(metricas)

    return pd.DataFrame(registros)

def graficar_metrica(df_metricas, columna, nombre_modelo='Modelo', ylabel=None, color='#722F37'):
    """
    Grafica la evolución de una métrica de verificación a lo largo de los
    timesteps del conjunto de validación, con la media como referencia.
    """
    serie = df_metricas[['idx', columna]].dropna(subset=[columna]) # <-- ignora NaN
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(serie['idx'], serie[columna], 'o-', color=color, markersize=4)

    media = serie[columna].mean()
    ax.axhline(media, color='gray', linestyle='--', alpha=0.6, label=f'Media: {media:.3f}')

    ax.set_xlabel('Timestep (val)')
    ax.set_ylabel(ylabel or columna.upper())
    ax.set_title(f'{ylabel or columna.upper()} — {nombre_modelo}')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"{columna}_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de {columna.upper()} guardada en: {ruta_guardado}")

    plt.show()
    return media

def graficar_mse(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'mse', nombre_modelo, ylabel='MSE')

def graficar_rmse(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'rmse', nombre_modelo, ylabel='RMSE (flashes/columna)')

def graficar_mae(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'mae', nombre_modelo, ylabel='MAE')

def graficar_ssim(df, nombre_modelo='Modelo'):
    return graficar_metrica(df, 'ssim', nombre_modelo, ylabel='SSIM')

def calcular_r2_global(modelo, dataset):
    """
    Calcula R² como métrica global sobre todos los píxeles activos
    del dataset.
    """
    modelo.eval()
    y_real_total = []
    y_pred_total = []

    for idx in tqdm(range(len(dataset)), desc="Calculando R² global"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = dataset.desnormalizar_y(y.squeeze()).numpy()
        y_pred = dataset.desnormalizar_y(pred_log).numpy()

        mascara = y_real > 0
        if mascara.sum() > 0:
            y_real_total.append(y_real[mascara])
            y_pred_total.append(y_pred[mascara])

    y_real_all = np.concatenate(y_real_total)
    y_pred_all = np.concatenate(y_pred_total)

    ss_res = np.sum((y_real_all - y_pred_all) ** 2)
    ss_tot = np.sum((y_real_all - y_real_all.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    print(f"R² global (sobre {len(y_real_all):,} píxeles activos): {r2:.4f}")
    return r2

def graficar_r2_comparativo(r2_dict, nombre_figura='r2_comparativo'):
    """
    Grafica un bar chart comparando el R² global entre modelos.

    Parámetros
    ----------
    r2_dict : dict {nombre_modelo: valor_r2}
        Ej: {'Baseline CNN': 0.72, 'U-Net': 0.85, 'Bob': 0.10}
    """
    nombres = list(r2_dict.keys())
    valores = list(r2_dict.values())
    colores = ['steelblue', '#722F37', 'seagreen'][:len(nombres)]

    fig, ax = plt.subplots(figsize=(7, 5))
    barras = ax.bar(nombres, valores, color=colores, edgecolor='black', width=0.5)

    # Línea de referencia en R²=0 (peor que predecir la media)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

    # Valor encima de cada barra
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width() / 2,
                valor + 0.01,
                f'{valor:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('R² (píxeles activos)')
    ax.set_title('R² Global por modelo')
    ax.set_ylim(min(0, min(valores)) - 0.1, max(valores) + 0.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    ruta_guardado = generar_ruta_imagen('../images', nombre_figura)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica guardada en: {ruta_guardado}")
    plt.show()

def graficar_picos_de_error(df_metricas, modelo, dataset, nombre_modelo='Modelo', top_n=3, metrica='mse', canal_entrada=0, vmax_percentil=99):
    """
    Encuentra los timesteps (índices) donde el modelo tuvo el mayor error 
    (los picos más altos) y grafica la reconstrucción para entender qué pasó.
    
    Parámetros
    ----------
    df_metricas   : DataFrame devuelto por evaluar_dataset().
    modelo        : modelo PyTorch ya cargado y en eval mode.
    dataset       : TormentaDataset (val_dataset).
    nombre_modelo : str, para el título.
    top_n         : int, cantidad de picos a mostrar (por defecto los 3 peores).
    metrica       : str, la columna del df a usar para buscar los picos ('mse', 'mae', etc).
    canal_entrada : int, índice del canal a mostrar en la gráfica de entrada.
    vmax_percentil : int, default 99
        Percentil de los valores activos (LIGHTDENS > 0) utilizado como
        límite superior de la escala de color en las reconstrucciones.
        Valores menores achican más la escala y resaltan mejor la estructura
        espacial a costa de saturar los picos más intensos.
    """
    # Verificamos que la métrica exista en el DataFrame
    if metrica not in df_metricas.columns:
        raise ValueError(f"La métrica '{metrica}' no se encuentra en el DataFrame.")

    # Calculamos la media para dar un poco de contexto de "qué tan grave" es el pico
    media_metrica = df_metricas[metrica].mean()
    
    # Extraemos las 'top_n' filas con los valores más altos de la métrica
    df_picos = df_metricas.nlargest(top_n, metrica)
    
    print(f"{"="*60}")
    print(f"🔍 ANALIZANDO LOS {top_n} PEORES CASOS SEGÚN {metrica.upper()}")
    print(f"Media global de {metrica.upper()} en este dataset: {media_metrica:.4f}")
    print(f"{"="*60}\n")
    
    # Iteramos sobre esas peores filas
    for ranking, (_, fila) in enumerate(df_picos.iterrows(), start=1):
        idx_pico = int(fila['idx'])
        valor_pico = fila[metrica]
        
        # Calculamos cuántas veces peor es este pico en comparación a la media
        multiplo = valor_pico / media_metrica if media_metrica > 0 else 0
        
        print(f"⚠️ PICO #{ranking} | Timestep: {idx_pico}")
        print(f"-> {metrica.upper()}: {valor_pico:.4f} (El error es {multiplo:.1f} veces la media global)")
        
        # Graficamos el índice extraído
        graficar_reconstruccion(
            modelo=modelo,
            dataset=dataset,
            nombre_modelo=nombre_modelo,
            canal_entrada=canal_entrada,
            idx=idx_pico,
            vmax_percentil=vmax_percentil
        )
        print("-" * 60)
        print("\n")

def resumen_pesos_loss(criterion):
    """
    Muestra los pesos aprendidos de Uncertainty Weighting al final
    del entrenamiento, convertidos a sigma^2 para interpretación directa.

    Valores más chicos de sigma^2 indican que ese término tuvo mayor
    peso relativo (más "confianza") durante el entrenamiento.
    """
    log_vars = {
        'WMSE': criterion.log_var_wmse.item(),
        'FAL':  criterion.log_var_fal.item(),
        'FCL':  criterion.log_var_fcl.item(),
    }

    print("PESOS APRENDIDOS (Uncertainty Weighting)")
    print("-" * 45)
    for nombre, log_var in log_vars.items():
        sigma2 = np.exp(log_var)
        precision = np.exp(-log_var)
        print(f"  {nombre:5s} — log_var: {log_var:+.4f} | σ²: {sigma2:.4f} | peso (precisión): {precision:.4f}")

    return log_vars

In [ ]:
archivos = sorted(glob.glob(f'../modelos/*{NOMBRE_MODELO}.pth'))
checkpoint = torch.load(archivos[-1], map_location=device)
modelo.load_state_dict(checkpoint['best_model_state_dict'])
modelo.eval()
resumen_entrenamiento(checkpoint, NOMBRE_MODELO, train_loader, val_loader, BATCH_SIZE)

In [ ]:
criterion.load_state_dict(checkpoint['criterion_state_dict'])
resumen_pesos_loss(criterion)

In [ ]:
graficar_curva_error(checkpoint, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_reconstruccion(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
animar_reconstruccion(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint)
ruta_txt = f"../modelos/historial_{NOMBRE_MODELO}.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo, val_dataset, data_range=data_range_val)

In [ ]:
graficar_mse(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_rmse(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_mae(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_ssim(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
r2 = calcular_r2_global(modelo, val_dataset)
graficar_r2_comparativo({f'{NOMBRE_MODELO}':r2,})

In [ ]:
graficar_picos_de_error(df_metricas, modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
modelo_best = UNet_WRF(in_channels=6, n_niveles=3, n_filtros_base=64, dropout_p=0.0, kernel_sizes=[5, 3, 3]).to(device)
modelo_best.load_state_dict(checkpoint['best_model_state_dict'])

modelo_final = UNet_WRF(in_channels=6, n_niveles=3, n_filtros_base=64, dropout_p=0.0, kernel_sizes=[5, 3, 3]).to(device)
modelo_final.load_state_dict(checkpoint['model_state_dict'])

for idx_test in range(279):
    X, y = val_dataset[idx_test]
    with torch.no_grad():
        pred_best  = modelo_best(X.unsqueeze(0).to(device)).cpu().squeeze()
        pred_final = modelo_final(X.unsqueeze(0).to(device)).cpu().squeeze()
    y_real = val_dataset.desnormalizar_y(y.squeeze()).numpy()
    y_best = val_dataset.desnormalizar_y(pred_best).numpy()
    y_fin  = val_dataset.desnormalizar_y(pred_final).numpy()
    print(f"idx={idx_test} | real_max={y_real.max():.1f} | best_max={y_best.max():.1f} | final_max={y_fin.max():.1f}")